# FDCPA Rule Classifier — QLoRA Training

**Run this on Kaggle with GPU (T4 x1 or T4 x2) enabled. Internet must be ON.**

- Base model: Qwen/Qwen2.5-3B-Instruct
- Quantization: NF4 (BitsAndBytes)
- LoRA: rank=16, alpha=32
- Expected runtime: ~60–90 min on T4

**Kaggle setup:**
1. Upload `data/train.jsonl` and `data/val.jsonl` as a Kaggle dataset named `fdcpa-data`
2. Add Kaggle Secrets: `HF_TOKEN`, `WANDB_API_KEY`

In [1]:
# Step 1: Clone repo and install dependencies
!git clone https://github.com/ree2raz/fdcpa-rule-classifier.git /kaggle/working/fdcpa-rule-classifier
!pip install -q transformers>=4.45.0 peft>=0.13.0 trl>=0.11.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets>=3.0.0 wandb>=0.18.0 python-dotenv tqdm scikit-learn matplotlib seaborn openai

import sys
sys.path.insert(0, '/kaggle/working/fdcpa-rule-classifier/src')
import os
os.chdir('/kaggle/working')

Cloning into '/kaggle/working/fdcpa-rule-classifier'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 32 (delta 3), reused 24 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 271.27 KiB | 12.33 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:
# Step 2: Copy data files from Kaggle dataset into repo
# Assumes you uploaded data/ as a Kaggle dataset named 'fdcpa-data'
import shutil
from pathlib import Path

repo_data = Path('/kaggle/working/fdcpa-rule-classifier/data')
kaggle_input = Path('/kaggle/input/datasets/ree2raz/fdcpa-data')

# If data exists as Kaggle dataset, copy it into the repo
if kaggle_input.exists():
    for f in kaggle_input.glob('*.jsonl'):
        shutil.copy2(f, repo_data / f.name)
        print(f'Copied {f.name}')
else:
    print('No Kaggle dataset found — data must already be in the repo or generated first.')

# Verify data exists
for split in ['train.jsonl', 'val.jsonl']:
    p = repo_data / split
    if p.exists():
        with open(p) as f:
            count = sum(1 for line in f if line.strip())
        print(f'{split}: {count} examples')
    else:
        print(f'WARNING: {split} not found at {p}')

Copied val.jsonl
Copied train.jsonl
train.jsonl: 208 examples
val.jsonl: 24 examples


## 1. Set Environment Variables

In [3]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

## 2. Initialize W&B

In [4]:
import wandb
from datetime import datetime

run_name = f"qlora-r16-a32-{datetime.now().strftime('%Y%m%d-%H%M')}"
wandb.init(
    project='fdcpa-rule-classifier',
    name=run_name,
    config={
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'lora_rank': 16,
        'lora_alpha': 32,
        'epochs': 3,
        'batch_size': 4,
        'gradient_accumulation': 4,
        'learning_rate': 2e-4,
    },
)
print(f"W&B run: {run_name}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bytebyte1001 (ree2raz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260503_145839-m2yb9yec
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run qlora-r16-a32-20260503-1458
wandb: ⭐️ View project at https://wandb.ai/ree2raz/fdcpa-rule-classifier
wandb: 🚀 View run at https://wandb.ai/ree2raz/fdcpa-rule-classifier/runs/m2yb9yec


W&B run: qlora-r16-a32-20260503-1458


## 3. Load Model and Tokenizer

In [5]:
import torch
from fdcpa_classifier.train import load_model_and_tokenizer, get_lora_config

model, tokenizer = load_model_and_tokenizer()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"VRAM reserved:  {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU: Tesla T4
VRAM allocated: 0.90 GB
VRAM reserved:  0.98 GB


## 4. Load and Tokenize Datasets

In [6]:
from fdcpa_classifier.train import build_dataset

train_ds = build_dataset('train', tokenizer)
val_ds = build_dataset('val', tokenizer)

print(f"Train: {len(train_ds)} examples")
print(f"Val:   {len(val_ds)} examples")

sample = train_ds[0]['text']
print(f"\nSample training example (first 500 chars):")
print(sample[:500] + '...')

Train: 208 examples
Val:   24 examples

Sample training example (first 500 chars):
<|im_start|>system
You are an FDCPA compliance evaluator. Given a debt collection rule and a transcript chunk from a collections call, determine whether the agent complied with the rule. Respond with a JSON object containing your verdict and reasoning.<|im_end|>
<|im_start|>user
## Rule
- **Rule ID:** FDCPA-002
- **Rule Name:** Validation of Debt
- **Description:** Upon written request within 30 days of initial communication, the debt collector must provide verification of the debt including the...


## 5. Configure LoRA

In [7]:
from peft import get_peft_model

lora_config = get_lora_config()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 6. Train

In [8]:
from fdcpa_classifier.train import get_training_args, MAX_SEQ_LENGTH
from trl import SFTTrainer

training_args = get_training_args()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer
)

import time
start_time = time.time()
trainer.train()
training_time = time.time() - start_time

print(f"\nTraining completed in {training_time / 60:.1f} minutes")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/208 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/208 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/24 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.762413,0.717502
100,0.607642,0.674182
150,0.496501,0.685472



Training completed in 47.5 minutes


## 7. Quality Checks

In [9]:
eval_results = trainer.evaluate()
print(f"Final eval loss: {eval_results.get('eval_loss', 'N/A')}")

train_loss = trainer.state.log_history
final_train_loss = [x['loss'] for x in train_loss if 'loss' in x]
if final_train_loss:
    print(f"Final train loss: {final_train_loss[-1]:.4f}")

if torch.cuda.is_available():
    peak_vram = torch.cuda.max_memory_allocated(0) / 1e9
    print(f"Peak VRAM: {peak_vram:.2f} GB")
    assert peak_vram < 14, f"VRAM exceeded 14GB: {peak_vram:.2f}GB"

print("\nAll quality checks passed.")

Final eval loss: 0.6741822361946106
Final train loss: 0.4965
Peak VRAM: 4.34 GB

All quality checks passed.


## 8. Save Adapter

In [10]:
from pathlib import Path

final_dir = Path('/kaggle/working/checkpoints/final')
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Adapter saved to {final_dir}")

Adapter saved to /kaggle/working/checkpoints/final


## 9. Push to HuggingFace Hub

In [11]:
hf_repo = 'ree2raz/fdcpa-rule-classifier-qlora'
model.push_to_hub(hf_repo, token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub(hf_repo, token=os.environ['HF_TOKEN'])
print(f"Pushed to https://huggingface.co/{hf_repo}")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/ree2raz/fdcpa-rule-classifier-qlora


In [12]:
wandb.finish()
print(f"\nTotal training time: {training_time / 60:.1f} minutes")
print(f"W&B run name: {run_name}")
print("Done. Proceed to 03_eval_comparison.ipynb")

wandb: updating run metadata
wandb: uploading summary, console lines 27-28
wandb: 
wandb: Run history:
wandb:             eval/entropy █▅▁▅
wandb:                eval/loss █▁▃▁
wandb: eval/mean_token_accuracy ▁▆█▆
wandb:          eval/num_tokens ▁▄██
wandb:             eval/runtime ▄▆▁█
wandb:  eval/samples_per_second ▅▄█▁
wandb:    eval/steps_per_second ▄▄█▁
wandb:            train/entropy ▇█▃▃▃▃▂▂▂▂▂▁▁▁▁
wandb:              train/epoch ▁▁▂▂▃▃▃▄▄▅▅▅▆▆▇▇████
wandb:        train/global_step ▁▁▂▂▃▃▃▄▄▅▅▅▆▆▇▇████
wandb:                       +5 ...
wandb: 
wandb: Run summary:
wandb:             eval/entropy 0.62195
wandb:                eval/loss 0.67418
wandb: eval/mean_token_accuracy 0.80574
wandb:          eval/num_tokens 257946
wandb:             eval/runtime 34.8165
wandb:  eval/samples_per_second 0.689
wandb:    eval/steps_per_second 0.172
wandb:               total_flos 4539082245488640.0
wandb:            train/entropy 0.50593
wandb:              train/epoch 3
wandb:              


Total training time: 47.5 minutes
W&B run name: qlora-r16-a32-20260503-1458
Done. Proceed to 03_eval_comparison.ipynb
